In [1]:
import os
import numpy as np
from orion.utils.o3d_utils import O3DPointCloud, scene_pcd_fn, visualize_o3d_point_cloud
from orion.utils.misc_utils import get_all_annotation_folder_info, quat_wxyz_to_xyzw, create_point_clouds_from_keypoints, invert_transformation_mat, plotly_draw_3d_pcd
from scipy.spatial.transform import Rotation as R
from scipy.ndimage import zoom
from PIL import Image
import torch


Jupyter environment detected. Enabling Open3D WebVisualizer.
[Open3D INFO] WebRTC GUI backend enabled.
[Open3D INFO] WebRTCWindowSystem: HTTP handshake server disabled.


[robosuite WARNING] No private macro file found! (macros.py:53)
[robosuite WARNING] It is recommended to use a private macro file (macros.py:54)
[robosuite WARNING] To setup, run: python /data/arisrei/miniconda3/envs/ORION/lib/python3.9/site-packages/robosuite/scripts/setup_macros.py (macros.py:55)


In [2]:
frame_idx = 0
annotation_folder = "/home/arisrei/research/RPL/ORION/annotations/human_demo/fugu_pufferrfish_up"
annotation_name = annotation_folder.split("/")[-1]
_, _, masks, K_dust3r = get_all_annotation_folder_info(annotation_folder)
show_track_proportion = .1

# align3r info
align3r_folder = os.path.join(annotation_folder, "align3r", annotation_name)
depth = np.load(os.path.join(align3r_folder, "all_depth_maps.npy"))
rgb = np.load(os.path.join(align3r_folder, "all_rgbs.npy"))
point_tracks = np.rint(
    torch.load(os.path.join(annotation_folder, "align3r", "tracked_points.pt"))[
        "pred_tracks"
    ]
    .detach()
    .cpu()
    .numpy()[0]
).astype(np.uint32)
intrinsics_list = np.loadtxt(
    os.path.join(align3r_folder, "pred_intrinsics.txt")
).reshape(-1, 3, 3)
camera_poses = np.loadtxt(os.path.join(align3r_folder, "pred_traj.txt")) # timestamp (1), xyz position (3), quaternion (4) wxyz
camera_poses = camera_poses[:, 1:]

FileNotFoundError: [Errno 2] No such file or directory: '/home/arisrei/research/RPL/ORION/annotations/human_demo/fugu_pufferrfish_up/align3r/tracked_points.pt'

In [ ]:
downsampled_masks = zoom(masks, (1, rgb.shape[1]/masks.shape[1], rgb.shape[2]/masks.shape[2]), order=0)
display(Image.fromarray((downsampled_masks[frame_idx] * 255).astype(np.uint8)))

In [6]:
camera_poses[:, 3:] = quat_wxyz_to_xyzw(camera_poses[:, 3:])
camera_quats = camera_poses[:, 3:]
camera_rotations = R.from_quat(camera_quats).as_matrix()
camera_translations = camera_poses[:, :3]
camera_pose_matrices = np.zeros((camera_poses.shape[0], 4, 4))
camera_pose_matrices[:, 3, 3] = 1
camera_pose_matrices[:, :3, :3] = camera_rotations
camera_pose_matrices[:, :3, 3] = camera_translations

# camera_pose_matrix_inv = np.zeros_like(camera_pose_matrices)
# camera_pose_matrix_inv[:, :3, :3] = np.linalg.inv(camera_pose_matrices[:, :3, :3])
# camera_pose_matrix_inv[:, :3, 3] = -np.linalg.inv(camera_pose_matrices[:, :3, :3]) @ camera_pose_matrices[:, :3, 3]

In [ ]:
keypoints_mask = np.zeros_like(downsampled_masks[frame_idx])
keypoints_mask[point_tracks[frame_idx][:, 1], point_tracks[frame_idx][:, 0]] = 1

first_frame_pcd_wrapper: O3DPointCloud = scene_pcd_fn(
    rgb[frame_idx], 
    depth[frame_idx], 
    intrinsics_list[frame_idx], 
    camera_pose_matrices[frame_idx],
    max_points=30000,
    is_real_robot=False,
    downsample=True,
    depth_trunc=5,
    depth_scale=1,
    return_pcd=True,
    mask=keypoints_mask,
    color_mask=True
)
fig = visualize_o3d_point_cloud(first_frame_pcd_wrapper, offline=True)
display(fig)

In [ ]:
all_points = np.zeros((rgb.shape[0], 30000, 3))
all_colors = np.zeros((rgb.shape[0], 30000, 3))

for i in range(rgb.shape[0]):
    pcd_wrapper: O3DPointCloud = scene_pcd_fn(
        rgb[i], 
        depth[i], 
        intrinsics_list[frame_idx], 
        #invert_transformation_mat(camera_pose_matrices[i]),
        np.eye(4),
        max_points=30000,
        is_real_robot=False,
        downsample=True,
        depth_trunc=5,
        depth_scale=1,
        return_pcd=True,
        mask=downsampled_masks[i],
        color_mask=True
    )
    all_points[i] = pcd_wrapper.get_points()
    all_colors[i] = pcd_wrapper.get_colors()

pcd_wrapper = O3DPointCloud(max_points=30000, points=all_points.reshape(-1, 3), colors=all_colors.reshape(-1, 3))
pcd_wrapper.preprocess()
fig = visualize_o3d_point_cloud(pcd_wrapper, offline=True)
display(fig)


    

In [ ]:
point_tracks_3d = np.zeros((point_tracks.shape[1], rgb.shape[0], 3))
for i in range(0, rgb.shape[0]):
    pcd_points, pcd_colors = create_point_clouds_from_keypoints(
        keypoints=point_tracks[i],
        depth_img=depth[i],
        camera_intrinsics_matrix=intrinsics_list[i],
        colors=rgb[i],
        scaling_factor=1
    )
    point_tracks_3d[:, i, :] = pcd_points

# pick some random points to draw
rand_idx = np.random.choice(point_tracks_3d.shape[0], int(point_tracks_3d.shape[0] * show_track_proportion), replace=False)
point_tracks_3d = point_tracks_3d[rand_idx]

fig = plotly_draw_3d_pcd(
    first_frame_pcd_wrapper.get_points(),
    first_frame_pcd_wrapper.get_colors(),
    addition_points=point_tracks_3d,
    offline=True,
    additional_point_draw_lines=True,
)
display(fig)

In [32]:
#fig.write_html("../../junk/grand_budapest_stir_butter_align3r.html")